In [7]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator
from data_preprocessing import create_train_test_val_sets, read_processed_data


In [16]:
#Create test train splits
x_mendeley, y_mendeley = read_processed_data(r"..\data\processed\mendeley_processed.csv")
x_phiusiil, y_phiusiil= read_processed_data(r"..\data\processed\phiusiil_processed.csv")

mendeley_sets = create_train_test_val_sets(x_mendeley,y_mendeley, label_col="Label", test_size=0.2, n_splits=5)
phiusiil_sets = create_train_test_val_sets(x_phiusiil,y_phiusiil, label_col="Label", test_size=0.2, n_splits=5)

Train/validation/test split prepared: 198360 instances for training and validation, 49590 instances for testing
Stratified 5-fold CV splits created.
Train/validation/test split prepared: 188636 instances for training and validation, 47159 instances for testing
Stratified 5-fold CV splits created.


### Tuning Classifiers

In [17]:
#XGBoost
def optimize_xgboost(X: pd.DataFrame, y: pd.Series, dataset: str) -> RandomizedSearchCV:
    scale_weights = [1.0]
    counts = y.value_counts(normalize=True)
    if dataset == 'mendeley':
        scale_weights.append(counts[1]/counts[0])
    elif dataset == 'phiusiil':
        scale_weights.append(counts[0]/counts[1])
    
    params = {
        'max_depth': [4, 5, 6, 8, 10],
        'min_child_weight': [1, 3, 5, 7],
        'gamma': [0, 0.1, 0.2, 0.4],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
        'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.4],
        'n_estimators': [100, 300, 500, 750, 1000],
        'scale_pos_weight': scale_weights
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    xgb = XGBClassifier(random_state=42)
    random_search = RandomizedSearchCV(xgb, param_distributions=params, random_state=42, cv=skf.split(X, y))
    random_search.fit(X, y)

    print('\n Best hyperparameters:')
    print(random_search.best_params_)

    return random_search

print("Running hyperparameter tuning using Mendeley Dataset:")
xgboost_mendeley = optimize_xgboost(mendeley_sets["x_train_val"], mendeley_sets["y_train_val"], 'mendeley')
print("Running hyperparameter tuning using Phiusiil Dataset:")
xgboost_phiusiil = optimize_xgboost(phiusiil_sets["x_train_val"], phiusiil_sets["y_train_val"], 'phiusiil')


Running hyperparameter tuning using Mendeley Dataset:

 Best hyperparameters:
{'subsample': 0.7, 'scale_pos_weight': np.float64(0.9289527680802856), 'n_estimators': 750, 'min_child_weight': 3, 'max_depth': 10, 'learning_rate': 0.4, 'gamma': 0.2, 'colsample_bytree': 0.8}
Running hyperparameter tuning using Phiusiil Dataset:

 Best hyperparameters:
{'subsample': 0.9, 'scale_pos_weight': 1.0, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.01, 'gamma': 0, 'colsample_bytree': 0.9}


In [25]:
from sklearn.metrics import mean_absolute_error, classification_report
def train_no_feature_selection(dataset, model):
    stratified_scores = []
    all_y_val = []
    all_y_pred = []
    #Training and Testing for both xgb classifiers
    for train_idx, val_idx in dataset["cv_splits"]:
        x_train, x_val = dataset["x_train_val"].iloc[train_idx], dataset["x_train_val"].iloc[val_idx]
        y_train, y_val = dataset["y_train_val"].iloc[train_idx], dataset["y_train_val"].iloc[val_idx]
        
        model.fit(x_train, y_train)
        y_pred = model.predict(x_val)
        stratified_scores.append(mean_absolute_error(y_val, y_pred))

        all_y_val.extend(y_val)
        all_y_pred.extend(y_pred)

    print(f"Mean MAE: {np.mean(stratified_scores):.4f}, Std MAE: {np.std(stratified_scores):.4f}")
    print('Confusion Matrix:')
    print(classification_report(all_y_val, all_y_pred))
    

print('Mendeley Results:')
train_no_feature_selection(mendeley_sets, xgboost_mendeley.best_estimator_)

print('Phiusiil Results:')
train_no_feature_selection(phiusiil_sets, xgboost_phiusiil.best_estimator_)



Mendeley Results:
Mean MAE: 0.0447, Std MAE: 0.0011
Confusion Matrix:
              precision    recall  f1-score   support

           0       0.95      0.96      0.96    102833
           1       0.96      0.95      0.95     95527

    accuracy                           0.96    198360
   macro avg       0.96      0.96      0.96    198360
weighted avg       0.96      0.96      0.96    198360

Phiusiil Results:
Mean MAE: 0.0000, Std MAE: 0.0000
Confusion Matrix:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     80756
           1       1.00      1.00      1.00    107880

    accuracy                           1.00    188636
   macro avg       1.00      1.00      1.00    188636
weighted avg       1.00      1.00      1.00    188636

